<div style="display: flex; align-items: center; gap: 16px;">
  
  <h1 style="color: #4A2D6F; font-weight: 800; margin: 0; flex: 1; text-align: center;">Microsoft Foundry: Agentic Observability — PoC</h1>
</div>
<p align="center" style="color: #666; font-style: italic; margin-top: -4px;">End-to-end proof of concept — from agent creation to trace validation in Log Analytics</p>

Build and interact with an AI Agent using Microsoft Foundry and the **Azure AI Projects SDK**.
The agent is configured for two tasks:

1. **Storytelling** — generate engaging stories from user prompts.
2. **Microsoft Learn via MCP** — query Foundry documentation through the Learn MCP endpoint.

For observability, the notebook wires up **Azure Monitor / OpenTelemetry tracing** to send agentic telemetry to:

- Microsoft Foundry — Tracing (Preview)
- Application Insights
- Interrogate Log Analytics (AppDependencies)

---

<h2 style="color: #D83B01;">Prerequisites</h2>

<details>
<summary><strong>Expand Prerequisites</strong></summary>

Before running this notebook, ensure the following are in place:

**1. Azure CLI Installed**

The notebook uses `DefaultAzureCredential`, which relies on Azure CLI for local authentication.

- **Install via winget (Windows):**
  ```powershell
  winget install --id Microsoft.AzureCLI -e --accept-source-agreements --accept-package-agreements
  ```
- **Other platforms / manual install:** [https://aka.ms/installazurecli](https://aka.ms/installazurecli)

**2. Logged in via Azure CLI with Appropriate Permissions**

After installing, sign in and verify your session:
```bash
az login
az account show
```
Your Entra ID identity must have **Contributor** (or equivalent) access to the Microsoft Foundry project and the associated Application Insights resource.

**3. Microsoft Foundry Project with Application Insights**

You need a project provisioned in **Microsoft Foundry** that is connected to an **Application Insights** instance backed by a **Log Analytics workspace**. The notebook retrieves the Application Insights connection string from the project at runtime to enable telemetry export.

</details>

---


<h2 style="color: #0078D4;">0. Create or Reuse Virtual Environment &amp; Register Kernel</h2>

<ul>
  <li>Create (or reuse) a Python virtual environment to isolate dependencies for this project, then install <code>ipykernel</code> and register it as a notebook kernel.</li>
  <li>Once setup is complete, select <strong>Kernel</strong> and choose your new <code>.venv</code> environment.</li>
</ul>


In [ ]:
import subprocess, sys, os

venv_dir = os.path.join(os.getcwd(), ".venv")

# Create the virtual environment (no-op if it already exists)
subprocess.check_call([sys.executable, "-m", "venv", venv_dir])

# Install / upgrade ipykernel inside the venv
pip = os.path.join(venv_dir, "bin", "pip")
subprocess.check_call([pip, "install", "--upgrade", "ipykernel"])

# Register the venv as a Jupyter kernel
python = os.path.join(venv_dir, "bin", "python")
subprocess.check_call([python, "-m", "ipykernel", "install", "--user", "--name", "ai-agent-demo", "--display-name", "AI Agent Demo (.venv)"])

print(f"✅ Virtual environment created or reused at {venv_dir}")
print("👉 Select the 'AI Agent Demo (.venv)' kernel in the top-right kernel picker, then continue.")

<h2 style="color: #0078D4;">1. Install Python Packages &amp; Dependencies</h2>

Install the required Python packages.

<details>
<summary><strong>Package Overview (Expand to view dependencies)</strong></summary>
<br>

📦 <code style="color: #0078D4; font-weight: 600;">azure-ai-projects</code>
 — SDK for working with Microsoft Foundry projects and agents.<br>
🔐 <code style="color: #0078D4; font-weight: 600;">azure-identity</code>
 — Provides <code>DefaultAzureCredential</code> for seamless Azure authentication via Entra ID.<br>
📡 <code style="color: #0E7C6B; font-weight: 600;">opentelemetry-sdk</code>
 — Core OpenTelemetry SDK for instrumentation and tracing.<br>
📊 <code style="color: #0078D4; font-weight: 600;">azure-monitor-opentelemetry</code>
 — All-in-one Azure Monitor OpenTelemetry distro (exporters, tracer providers, etc.).<br>
&emsp;↳ <code style="color: #888; font-weight: 500;">azure-core-tracing-opentelemetry</code>
 <span style="color: #666; font-size: 0.9em;">Transitive dep — Azure SDK tracing plugin.</span>

</details>


In [ ]:
%pip install --no-input --pre "azure-ai-projects>=2.0.0b4" azure-identity "opentelemetry-sdk<1.39" "opentelemetry-api<1.39" azure-monitor-opentelemetry azure-core-tracing-opentelemetry

<h2 style="color: #0078D4;">2. Import Libraries</h2>

Import the necessary classes from the Azure, Foundry, and OpenTelemetry SDKs.

<details>
<summary><strong>What Each Import Does</strong></summary>
<br>

🔐 <code style="color: #0078D4; font-weight: 600;">DefaultAzureCredential</code>
 <span style="color: #888; font-size: 0.88em;">azure-identity</span>
 — Picks up credentials from your environment (Azure CLI, managed identity, etc.).<br>
🤖 <code style="color: #0078D4; font-weight: 600;">AIProjectClient</code>
 <span style="color: #888; font-size: 0.88em;">azure-ai-projects</span>
 — Client for your Foundry project (setup, agent creation, telemetry connection string).<br>
✨ <code style="color: #0078D4; font-weight: 600;">PromptAgentDefinition</code>
 <span style="color: #888; font-size: 0.88em;">azure-ai-projects</span>
 — Defines the agent's model and behavior instructions.<br>
📡 <code style="color: #0E7C6B; font-weight: 600;">AIProjectInstrumentor</code>
 <span style="color: #888; font-size: 0.88em;">azure-ai-projects</span>
 — Instruments SDK and OpenAI operations for tracing (Section 3.1).<br>
📊 <code style="color: #0E7C6B; font-weight: 600;">configure_azure_monitor</code>
 <span style="color: #888; font-size: 0.88em;">azure-monitor-opentelemetry</span>
 — One-liner to set up Azure Monitor as the OpenTelemetry export backend.

</details>

<details>
<summary><strong>How the Pieces Wire Together</strong></summary>
<br>

The notebook connects the Foundry SDK to the telemetry stack in four steps:<br>

<span style="color: #0078D4; font-weight: 700;">①</span>
 <code style="color: #0078D4;">project_client.telemetry.get_application_insights_connection_string()</code>
 — retrieves the destination from the Foundry project.<br>
<span style="color: #0078D4; font-weight: 700;">②</span>
 <code style="color: #0078D4;">configure_azure_monitor(...)</code>
 — initializes the Azure Monitor OpenTelemetry pipeline.<br>
<span style="color: #0078D4; font-weight: 700;">③</span>
 <code style="color: #0E7C6B;">AIProjectInstrumentor().instrument(...)</code>
 — instruments <code>azure-ai-projects</code> and OpenAI operations.<br>
<span style="color: #0078D4; font-weight: 700;">④</span>
 Manual spans (<code style="color: #D83B01;">create_agent</code>, <code style="color: #D83B01;">invoke_agent</code>)
 — add business context for troubleshooting.<br>

<table style="border: none; border-collapse: collapse; margin-top: 4px;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🏗️</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Foundry SDK</strong> — project setup, agent creation/query, connection string retrieval.</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📡</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Telemetry Stack</strong> — <code>opentelemetry-sdk</code> (tracing runtime) → <code>azure-core-tracing-opentelemetry</code> (bridge) → <code>azure-monitor-opentelemetry</code> (exporter).</td>
  </tr>
</table>

</details>

<details>
<summary><strong>MSFT Learn References</strong></summary>

- [Microsoft Foundry tracing flow](https://learn.microsoft.com/azure/foundry-classic/how-to/develop/trace-agents-sdk#instrument-tracing-in-your-code)
- [Azure Monitor ()](https://learn.microsoft.com/python/api/overview/azure/monitor-opentelemetry-readme?view=azure-python#getting-started)
- [Azure Core tracing bridge](https://learn.microsoft.com/python/api/overview/azure/core-tracing-opentelemetry-readme?view=azure-python-preview#getting-started)

</details>


In [1]:
import os, sys, platform
from azure.core.settings import settings

# Force Azure SDKs to emit OpenTelemetry spans
settings.tracing_implementation = "opentelemetry"

# Set resource detector override BEFORE importing azure-monitor-opentelemetry
# to ensure it takes effect during module initialization
if platform.system() == "Darwin":
    os.environ["OTEL_EXPERIMENTAL_RESOURCE_DETECTORS"] = "otel"
    os.environ["OTEL_RESOURCE_DETECTORS"] = "otel"
    print(f"⚙️  macOS detected — disabled Azure IMDS resource detector to avoid local hang")

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.ai.projects.telemetry import AIProjectInstrumentor

print("✅ All libraries imported successfully")
print(f"   💻 Platform: {platform.system()} {platform.release()}")
print(f"   🐍 Python:   {sys.version.split()[0]}")

⚙️  macOS detected — disabled Azure IMDS resource detector to avoid local hang
✅ All libraries imported successfully
   💻 Platform: Darwin 25.3.0
   🐍 Python:   3.14.3


<h2 style="color: #0078D4;">3. Configure the Project Client</h2>

- Reuse the deployment values loaded in the **Confirm Existing Deployment** section above.
- Create an authenticated `AIProjectClient` instance.
- Confirm which credential was selected for this notebook session.


In [2]:
import json
import logging
import subprocess
from pathlib import Path

def resolve_build_info_path() -> Path:
    candidates = sorted(
        Path(".").glob("build_info-*.json"),
        key=lambda candidate: candidate.stat().st_mtime,
        reverse=True,
    )
    if candidates:
        return candidates[0]

    raise FileNotFoundError(
        r"No build_info-<suffix>.json file found. Run deployment/deploy-foundry-env.ps1 first to generate one."
    )

build_info_path = resolve_build_info_path()
build_info = json.loads(build_info_path.read_text(encoding="utf-8"))
foundry_proj_ep = build_info["foundry_project_endpoint"]
genai_model = build_info["genai_model"]
signed_in_account = globals().get("signed_in_account")

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=foundry_proj_ep,
    credential=credential,
)

_lw = 14
print("\n✅ AIProjectClient Configured")
print(f"   🌐 {'Endpoint:':<{_lw}} {foundry_proj_ep}")
print(f"   🧠 {'Model:':<{_lw}} {genai_model}")
print(f"   📄 {'Build info:':<{_lw}} {build_info_path.resolve()}")
print(f"   🔐 {'Auth:':<{_lw}} DefaultAzureCredential")

def _run_command(command: list[str]) -> str | None:
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
        if result.returncode == 0:
            value = (result.stdout or "").strip()
            return value or None
    except Exception:
        pass
    return None

def _resolve_identity_hint(credential_name: str) -> str | None:
    if credential_name == "AzureCliCredential":
        return _run_command(["az", "account", "show", "--query", "user.name", "-o", "tsv"])
    if credential_name == "AzurePowerShellCredential":
        powershell_cmd = "$ctx = Get-AzContext; if ($ctx -and $ctx.Account) { $ctx.Account.Id }"
        return _run_command(["pwsh", "-NoProfile", "-Command", powershell_cmd])
    return None

try:
    identity_logger = logging.getLogger("azure.identity")
    if not identity_logger.handlers:
        identity_logger.addHandler(logging.StreamHandler())
    identity_logger.setLevel(logging.INFO)

    _ = credential.get_token("https://management.azure.com/.default")
    selected_credential = getattr(credential, "_successful_credential", None)
    if selected_credential is not None:
        credential_name = selected_credential.__class__.__name__
        print(f"   🔐 Credential used: {credential_name}")

        identity_hint = _resolve_identity_hint(credential_name)
        if identity_hint:
            signed_in_account = identity_hint
            globals()["signed_in_account"] = signed_in_account
            print(f"   👤 Signed-in account: {identity_hint}")
        else:
            print("   👤 Signed-in account: not available for this credential type")
    else:
        print("   🔍 Credential used: check azure.identity INFO logs above")
except Exception as ex:
    print(f"   ⚠️ Credential probe skipped: {type(ex).__name__}: {ex}")


✅ AIProjectClient Configured
   🌐 Endpoint:      https://zolabai-foundry-nhiv0q.services.ai.azure.com/api/projects/zolabai-fndry-proj-nhiv0q
   🧠 Model:         gpt-5.3
   📄 Build info:    /Users/lorenzo/agentic_observability/MSFT_Foundry_Agent_Telemetry_Demo/build_info-nhiv0q.json
   🔐 Auth:          DefaultAzureCredential


DefaultAzureCredential acquired a token from AzureCliCredential


   🔐 Credential used: AzureCliCredential
   👤 Signed-in account: lireland@DibSecurity.onmicrosoft.com


<h2 style="color: #0078D4;">3.1 Enable AI Agent (client-side) Observability/Telemetry</h2>

<details>
<summary><strong>Tracing Setup Overview</strong></summary>

Configure tracing per the [Azure AI Projects SDK tracing guide](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/ai/azure-ai-projects#tracing):
1. **`configure_azure_monitor`** — Sets up the OpenTelemetry trace exporter that sends spans to Application Insights
2. **`AIProjectInstrumentor`** — Instruments `azure-ai-projects` SDK operations and OpenAI responses/conversations operations

**Note:**
- `AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING=true` must be set **before** calling `instrument()`.
- `OTEL_SEMCONV_STABILITY_OPT_IN=gen_ai_latest_experimental` opts into the latest experimental GenAI semantic conventions.
- Content recording is controlled by `OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT`.
- Trace context propagation and baggage propagation are controlled by the `AZURE_TRACING_GEN_AI_*` environment variables set below.
- This notebook configures Azure Monitor in **trace-only** mode on local machines to avoid the Azure Monitor metrics/performance-counter startup path that can stall on macOS/Python 3.14.

</details>

In [3]:
import os
import platform
from uuid import uuid4

from azure.core.settings import settings
from opentelemetry import baggage, context as otel_context, trace
from opentelemetry.sdk.resources import Resource
from azure.monitor.opentelemetry import configure_azure_monitor

# Keep Azure SDK tracing disabled until Azure Monitor is configured so the
# connection-string lookup below does not inherit a stale global provider from
# a previous failed run in the current notebook kernel.
settings.tracing_implementation = None

# ---------------------------------------------------------------------------
# Phase 1: Trace settings (must be set before instrumentation)
# ---------------------------------------------------------------------------
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"
os.environ["OTEL_SEMCONV_STABILITY_OPT_IN"] = "gen_ai_latest_experimental"
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "true"
os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true"
os.environ["AZURE_TRACING_GEN_AI_ENABLE_TRACE_CONTEXT_PROPAGATION"] = "true"
os.environ["AZURE_TRACING_GEN_AI_TRACE_CONTEXT_PROPAGATION_INCLUDE_BAGGAGE"] = "true"

# macOS can stall on Azure host resource detection when running locally
if platform.system() == "Darwin":
    os.environ["OTEL_EXPERIMENTAL_RESOURCE_DETECTORS"] = "otel"
    os.environ["OTEL_RESOURCE_DETECTORS"] = "otel"

project_name = foundry_proj_ep.rstrip("/").split("/")[-1] if "foundry_proj_ep" in globals() else "unknown-project"
os.environ["OTEL_SERVICE_NAME"] = f"foundry-ai-agent-1702"
os.environ["OTEL_TRACES_SAMPLER"] = "microsoft.fixed.percentage"
os.environ["OTEL_TRACES_SAMPLER_ARG"] = "1.0"

telemetry_session_id = globals().get("telemetry_session_id")
if not telemetry_session_id:
    telemetry_session_id = str(uuid4())
    globals()["telemetry_session_id"] = telemetry_session_id

# -----------------------------------------------------------------------------
# Phase 2: Backend setup (Microsoft Foundry -> Application Insights connection)
# -----------------------------------------------------------------------------
application_insights_connection_string = project_client.telemetry.get_application_insights_connection_string()

resource = Resource.create(
    {
        "service.name": os.environ["OTEL_SERVICE_NAME"],
        "service.namespace": "foundry-agent-1702",
        "service.instance.id": telemetry_session_id,
        "deployment.environment": "demo",
        "foundry.project.name": project_name,
    }
)

# Configure Azure Monitor as a trace-only backend for local notebook stability.
# The metrics/performance-counter startup path can stall on macOS/Python 3.14.
configure_azure_monitor(
    connection_string=application_insights_connection_string,
    resource=resource,
    sampling_ratio=1.0,
    disable_logging=True,
    disable_metrics=True,
    enable_performance_counters=False,
 )

# Azure SDKs should emit OpenTelemetry spans only after the trace backend exists.
settings.tracing_implementation = "opentelemetry"

# -----------------------------------------------------------------------------
# Phase 3: SDK instrumentation
# -----------------------------------------------------------------------------
try:
    AIProjectInstrumentor().uninstrument()
except Exception:
    pass

# Content recording is passed explicitly; trace/baggage propagation is read from
# the AZURE_TRACING_GEN_AI_* environment variables above to avoid version-specific
# keyword argument mismatches across azure-ai-projects builds.
AIProjectInstrumentor().instrument(
    enable_content_recording=True,
 )

# -----------------------------------------------------------------------------
# Phase 4: Tracer handle for custom spans in later sections
# -----------------------------------------------------------------------------
tracer = trace.get_tracer(__name__)

def make_baggage_context(values: dict[str, object]):
    context = otel_context.get_current()
    for key, value in values.items():
        if value is None:
            continue
        value_text = str(value).strip()
        if value_text:
            context = baggage.set_baggage(key, value_text, context=context)
    return context

ansi_cyan = ""
ansi_magenta = ""
ansi_reset = ""
project_name_color = ansi_magenta
project_name_colored = f"{project_name_color}{project_name}{ansi_reset}"
foundry_label_colored = f"{project_name_color}Microsoft Foundry{ansi_reset}"

print(f"Tracing enabled -> Application Insights for project: '{project_name_colored}'")
print(f"- [connection string retrieved from: {foundry_label_colored}]")
print(f"- OTEL service name: {project_name_color}{os.environ['OTEL_SERVICE_NAME']}{ansi_reset}")
print(f"- Telemetry session id: {telemetry_session_id}")
print("- Trace context propagation: enabled")
print("- Baggage propagation: enabled")
print("- Content recording: enabled")
print("- Azure Monitor mode: traces only (logging/metrics disabled for local stability)")
print("- Trace sampling: 100%")

AzureCliCredential.get_token_info succeeded
DefaultAzureCredential acquired a token from AzureCliCredential


Tracing enabled -> Application Insights for project: 'zolabai-fndry-proj-nhiv0q'
- [connection string retrieved from: Microsoft Foundry]
- OTEL service name: foundry-ai-agent-1702
- Telemetry session id: 29ef86d4-6c09-4c82-8767-140399e09118
- Trace context propagation: enabled
- Baggage propagation: enabled
- Content recording: enabled
- Azure Monitor mode: traces only (logging/metrics disabled for local stability)
- Trace sampling: 100%


<h2 style="color: #0078D4;">4. Create the AI Agent</h2>

<table style="border: none; border-collapse: collapse; margin-top: 4px;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">✨</td>
e    <td style="border: none; padding: 4px 0; color: #555;">Storytelling persona — crafts engaging 6-sentence stories from user prompts while also acting as a Microsoft technologies learning assistant</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🧠</td>
    <td style="border: none; padding: 4px 0; color: #555;">Model comes from the deployment metadata loaded earlier via <code style="color: #0078D4;">genai_model</code>, not a hard-coded model name</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🔧</td>
    <td style="border: none; padding: 4px 0; color: #555;">MSFT Learn MCP tool is attached for Foundry documentation lookups; if the tool spec was not created earlier, this step bootstraps the default endpoint</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📡</td>
    <td style="border: none; padding: 4px 0; color: #555;">Agent creation span now carries baggage and GenAI semantic attributes for stronger trace correlation</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🔄</td>
    <td style="border: none; padding: 4px 0; color: #555;"><code style="color: #0078D4;">create_version</code> is idempotent — only bumps the version when parameters change</td>
  </tr>
</table>


In [7]:
from uuid import uuid4
from opentelemetry.trace import SpanKind, Status, StatusCode

agent_name = "ZoDEfendersAgent-1702"
model_name = genai_model

agent_instructions = (
    "You are an illuminating cybernetic storytelling agent. "
    "You craft engaging 6-sentence stories based on user prompts and context. "
    "Be sure to use vivid language and creative scenarios to captivate the reader. "
    "Use the full names of any characters you create in the story. "
    "Lastly, you are also a helpful assistant for learning about Microsoft technologies. "
    "If asked, only employ the tools available to answer questions and provide relevant topic information."
)

mcp_tool = globals().get("mcp_tool_spec")
if mcp_tool is None:
    from azure.ai.projects.models import MCPTool

    msft_learn_mcp_url = "https://learn.microsoft.com/api/mcp"
    mcp_tool = MCPTool(
        server_label="msft-learn",
        server_url=msft_learn_mcp_url,
    )
    globals()["mcp_tool_spec"] = mcp_tool

# Shared run correlation id across notebook steps (created once per kernel session)
demo_run_id = globals().get("demo_run_id")
if demo_run_id is None:
    demo_run_id = str(uuid4())
    globals()["demo_run_id"] = demo_run_id

tool_labels = [getattr(mcp_tool, "server_label", "unknown-tool")]
agent_creation_context = make_baggage_context(
    {
        "demo-run-id": demo_run_id,
        "agent-name": agent_name,
        "model-name": model_name,
        "project-name": globals().get("project_name", "unknown-project"),
        "telemetry-session-id": globals().get("telemetry_session_id", ""),
    }
)

# OTel GenAI Agent semconv-compliant create span
with tracer.start_as_current_span(
    f"create_agent {agent_name}",
    kind=SpanKind.CLIENT,
    context=agent_creation_context,
 ) as span:
    span.set_attribute("gen_ai.operation.name", "create_agent")
    span.set_attribute("gen_ai.provider.name", "azure.ai.openai")
    span.set_attribute("gen_ai.request.model", model_name)
    span.set_attribute("gen_ai.agent.name", agent_name)
    span.set_attribute("gen_ai.agent.description", agent_instructions)
    span.set_attribute("demo.run_id", demo_run_id)
    span.set_attribute("app.session.id", globals().get("telemetry_session_id", ""))
    span.set_attribute("agent.tools.count", len(tool_labels))
    span.set_attribute("agent.tools.labels", ",".join(tool_labels))
    span.add_event("create_agent.start")

    try:
        # Creates an agent, bumps the agent version if parameters have changed
        agent = project_client.agents.create_version(
            agent_name=agent_name,
            definition=PromptAgentDefinition(
                model=model_name,
                instructions=agent_instructions,
                tools=[mcp_tool],
            ),
        )
        span.set_attribute("gen_ai.agent.id", agent.id)
        span.set_attribute("gen_ai.agent.version", str(agent.version))
        span.add_event("create_agent.success")
    except Exception as ex:
        span.record_exception(ex)
        span.set_attribute("error.type", type(ex).__name__)
        span.set_status(Status(StatusCode.ERROR, str(ex)))
        span.add_event("create_agent.failure")
        raise

agent_name_color = globals().get("ansi_magenta", "")
agent_ansi_reset = globals().get("ansi_reset", "")
agent_name_colored = f"{agent_name_color}{agent.name}{agent_ansi_reset}"

print(f"Agent created: {agent_name_colored} (id: {agent.id}, version: {agent.version})")
print(f"Run correlation id: {demo_run_id}")

Agent created: ZoDEfendersAgent-1702 (id: ZoDEfendersAgent-1702:2, version: 2)
Run correlation id: 70529c00-aab2-4fa6-b6a0-8359e0fd328d


<h2 style="color: #0078D4;">5. Query the Agent</h2>

<table style="border: none; border-collapse: collapse; margin-top: 4px;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📖</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Pass 1</strong> — generate a fictional story in its own Foundry conversation for clean trace separation</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🔍</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Pass 2</strong> — retrieve factual guidance from MSFT Learn via the MCP tool in a second independent conversation</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🧾</td>
    <td style="border: none; padding: 4px 0; color: #555;">Conversation IDs, approval rounds, and combined output are captured so traces and persisted results line up</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">💾</td>
    <td style="border: none; padding: 4px 0; color: #555;">Story output and MSFT Learn insights are appended to <code style="color: #2EA043;">stories.json</code> with environment metadata</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📡</td>
    <td style="border: none; padding: 4px 0; color: #555;">Each interaction and persistence step is wrapped in <code style="color: #0E7C6B;">OpenTelemetry</code> spans with baggage for correlation</td>
  </tr>
</table>


In [9]:
import json
from datetime import datetime, timezone
from pathlib import Path

from opentelemetry import context as otel_context
from opentelemetry.trace import SpanKind

story_prompt = (
    "Write about a Cloud Solutions Architect - Security by the name of 'Azure Zo' in Frankfurt, Germany. "
    "He is a man who moonlights as a cybernetic superhero called 'Die DEfender', protector of the digital realm. "
    "Include a plot twist in the last sentence. Always refer to the two characters by their full names."
)

facts_prompt = (
    "Use the MSFT Learn MCP tool to explain what Microsoft Foundry is for a Cloud Solutions Architect - Security. "
    "Return concise factual guidance only with this exact structure:\n"
    "MSFT Learn Insights:\n"
    "- 3 to 5 key points\n"
    "- Security and governance considerations\n"
    "- Practical next steps\n"
    "Sources:\n"
    "- Include at least one Microsoft Learn URL"
)

MAX_APPROVAL_ROUNDS = 5
agent_reference_payload = {
    "agent_reference": {
        "name": agent.name,
        "id": agent.id,
        "type": "agent_reference",
    }
}
conversation_ids: dict[str, str] = {}
run_id = str(globals().get("demo_run_id", "") or "")
session_id = str(globals().get("telemetry_session_id", "") or "")
project_name_value = str(globals().get("project_name", "unknown-project") or "unknown-project")
agent_version = str(getattr(agent, "version", "") or "")

with project_client.get_openai_client() as openai_client:
    def parse_response(response) -> tuple[str, list[str], list[str], object]:
        output_items = getattr(response, "output", None) or []
        output_types = [getattr(item, "type", type(item).__name__) for item in output_items]

        approval_ids = [
            item.id
            for item in output_items
            if getattr(item, "type", None) == "mcp_approval_request" and getattr(item, "id", None)
        ]

        text_parts = []
        for item in output_items:
            for content in getattr(item, "content", None) or []:
                text_obj = getattr(content, "text", None)
                if isinstance(text_obj, str) and text_obj.strip():
                    text_parts.append(text_obj)
                    continue

                text_value = getattr(text_obj, "value", None)
                if isinstance(text_value, str) and text_value.strip():
                    text_parts.append(text_value)

        text = (getattr(response, "output_text", None) or "\n".join(text_parts)).strip()
        return text, approval_ids, output_types, getattr(response, "incomplete_details", None)

    def run_query_with_auto_approval(openai_client, agent_reference_payload, prompt, interaction_name, interaction_span, max_rounds=5):
        conversation = openai_client.conversations.create()
        print(f"Conversation created: {conversation.id}")
        interaction_span.add_event(
            "conversation.created",
            {
                "app.conversation.id": conversation.id,
                "app.interaction": interaction_name,
            },
        )

        baggage_values = {
            "demo-run-id": run_id,
            "agent-name": agent.name,
            "agent-id": agent.id,
            "agent-version": agent_version,
            "interaction-name": interaction_name,
            "conversation-id": conversation.id,
            "model-name": model_name,
            "telemetry-session-id": session_id,
        }

        def create_agent_response(response_input):
            baggage_token = otel_context.attach(make_baggage_context(baggage_values))
            try:
                return openai_client.responses.create(
                    conversation=conversation.id,
                    input=response_input,
                    extra_body=agent_reference_payload,
                )
            finally:
                otel_context.detach(baggage_token)

        response = create_agent_response(prompt)
        approvals_granted = 0

        for round_number in range(1, max_rounds + 1):
            text, approval_ids, output_types, incomplete_details = parse_response(response)
            status = str(getattr(response, "status", "unknown") or "unknown")
            response_id = str(getattr(response, "id", "") or "")

            print(f"Response status: {status}")
            print(f"Response output types: {output_types}")
            if incomplete_details:
                print(f"Incomplete details: {incomplete_details}")
                interaction_span.add_event(
                    "response.incomplete",
                    {"app.incomplete_details": str(incomplete_details)},
                )

            interaction_span.set_attribute("app.response.status", status)
            interaction_span.set_attribute("app.response.output_types", ",".join(output_types))

            if text or not approval_ids:
                interaction_span.add_event(
                    "response.completed",
                    {
                        "gen_ai.response.id": response_id,
                        "app.approval.rounds": approvals_granted,
                    },
                )
                return conversation.id, response, text, approvals_granted

            approvals_granted += len(approval_ids)
            interaction_span.add_event(
                "mcp.approval.auto_approved",
                {
                    "app.approval.round": round_number,
                    "app.approval.count": len(approval_ids),
                },
            )
            print(f"Approving MCP tool requests (round {round_number}): {approval_ids}")
            response = create_agent_response(
                [
                    {
                        "type": "mcp_approval_response",
                        "approve": True,
                        "approval_request_id": request_id,
                    }
                    for request_id in approval_ids
                ]
            )

        final_text, _, _, _ = parse_response(response)
        interaction_span.add_event(
            "response.max_approval_rounds_reached",
            {"app.approval.rounds": approvals_granted},
        )
        print("⚠️ Reached maximum MCP approval rounds without assistant text output.")
        return conversation.id, response, final_text, approvals_granted

    def run_interaction_with_span(*, interaction_name: str, prompt: str):
        interaction_context = make_baggage_context(
            {
                "demo-run-id": run_id,
                "agent-name": agent.name,
                "agent-id": agent.id,
                "agent-version": agent_version,
                "interaction-name": interaction_name,
                "model-name": model_name,
                "project-name": project_name_value,
                "telemetry-session-id": session_id,
            }
        )
        with tracer.start_as_current_span(
            f"invoke_agent {agent.name}",
            kind=SpanKind.CLIENT,
            context=interaction_context,
        ) as interaction_span:
            if run_id:
                interaction_span.set_attribute("demo.run_id", run_id)
            interaction_span.set_attribute("gen_ai.operation.name", "invoke_agent")
            interaction_span.set_attribute("gen_ai.provider.name", "azure.ai.openai")
            interaction_span.set_attribute("gen_ai.request.model", model_name)
            interaction_span.set_attribute("gen_ai.agent.name", agent.name)
            interaction_span.set_attribute("gen_ai.agent.id", agent.id)
            interaction_span.set_attribute("gen_ai.agent.version", agent_version)
            interaction_span.set_attribute("app.interaction", interaction_name)
            interaction_span.set_attribute("app.prompt", prompt)
            interaction_span.set_attribute("app.session.id", session_id)

            conversation_id, response, text, approvals_granted = run_query_with_auto_approval(
                openai_client=openai_client,
                agent_reference_payload=agent_reference_payload,
                prompt=prompt,
                interaction_name=interaction_name,
                interaction_span=interaction_span,
                max_rounds=MAX_APPROVAL_ROUNDS,
            )

            conversation_ids[interaction_name] = conversation_id
            interaction_span.set_attribute("app.conversation.id", conversation_id)
            interaction_span.set_attribute("app.approval.rounds", approvals_granted)
            interaction_span.set_attribute("app.completion", text[:500] if text else "")
            interaction_span.set_attribute("gen_ai.response.id", str(getattr(response, "id", "") or ""))
            response_model = getattr(response, "model", None)
            if response_model:
                interaction_span.set_attribute("gen_ai.response.model", str(response_model))
            return response, text

    run_context = make_baggage_context(
        {
            "demo-run-id": run_id,
            "agent-name": agent.name,
            "agent-id": agent.id,
            "model-name": model_name,
            "project-name": project_name_value,
            "telemetry-session-id": session_id,
        }
    )
    with tracer.start_as_current_span("agent-query", context=run_context) as run_span:
        if run_id:
            run_span.set_attribute("demo.run_id", run_id)
        run_span.set_attribute("agent.name", agent.name)
        run_span.set_attribute("gen_ai.request.model", model_name)
        run_span.set_attribute("app.session.id", session_id)

        print("\n=== Pass 1: Generate fictional story (independent conversation) ===")
        story_response, story_text = run_interaction_with_span(
            interaction_name="story",
            prompt=story_prompt,
        )

        print("\n=== Pass 2: Retrieve factual MSFT Learn insights (independent conversation) ===")
        facts_response, facts_text = run_interaction_with_span(
            interaction_name="facts",
            prompt=facts_prompt,
        )

        output_sections = []
        if story_text:
            output_sections.append("Story:\n" + story_text.strip())
        if facts_text:
            output_sections.append(facts_text.strip())

        assistant_text = "\n\n".join(section for section in output_sections if section).strip()
        run_span.set_attribute("app.output.sections", len(output_sections))
        run_span.set_attribute("app.conversation.count", len(conversation_ids))

if not assistant_text:
    print("⚠️ No assistant text was returned.")

print(f"\nResponse output:\n{assistant_text}")
if conversation_ids:
    print("\nFoundry conversation IDs:")
    for interaction_name, conversation_id in conversation_ids.items():
        print(f"- {interaction_name}: {conversation_id}")

def append_story(stories_path: Path, payload: dict) -> int:
    try:
        with stories_path.open("r", encoding="utf-8") as file:
            stories = json.load(file)
        if not isinstance(stories, list):
            stories = []
    except (FileNotFoundError, json.JSONDecodeError):
        stories = []

    next_id = max((item.get("id", 0) for item in stories), default=0) + 1
    payload["id"] = next_id
    stories.append(payload)

    with stories_path.open("w", encoding="utf-8") as file:
        json.dump(stories, file, indent=2, ensure_ascii=False)

    return next_id

stories_file = Path("stories.json")
persist_context = make_baggage_context(
    {
        "demo-run-id": run_id,
        "agent-name": agent.name,
        "model-name": model_name,
        "project-name": project_name_value,
        "telemetry-session-id": session_id,
    }
)
with tracer.start_as_current_span("persist_story", context=persist_context) as persist_span:
    story_id = append_story(
        stories_file,
        {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "agent": agent.name,
            "model": model_name,
            "prompt": story_prompt,
            "story": story_text,
            "msft_learn_insights": facts_text,
            "combined_output": assistant_text,
            "foundry_project_endpoint": build_info["foundry_project_endpoint"],
            "resource_group": build_info["rg"],
            "requested_by": build_info.get("requested_by"),
            "conversation_ids": conversation_ids,
        },
    )
    persist_span.set_attribute("app.story.id", story_id)
    persist_span.set_attribute("app.stories.path", str(stories_file))
    persist_span.set_attribute("app.conversation.count", len(conversation_ids))

print(f"\nStory #{story_id} saved to {stories_file}")


=== Pass 1: Generate fictional story (independent conversation) ===


AzureCliCredential.get_token_info succeeded
DefaultAzureCredential acquired a token from AzureCliCredential


Conversation created: conv_f9d0e5863a2588b200hZac1bTCf5qp03KOTreI6RlTbyoxv8GQ
Response status: completed
Response output types: ['mcp_list_tools', 'reasoning', 'message']

=== Pass 2: Retrieve factual MSFT Learn insights (independent conversation) ===
Conversation created: conv_7b47cf352d75ef4d00XRiBqcQOtdh5Z7Hgkz6lq3litdOGBR4g
Response status: completed
Response output types: ['mcp_list_tools', 'mcp_approval_request']
Approving MCP tool requests (round 1): ['mcpr_7b47cf352d75ef4d0069b084f9bb848190804a1e8a7e7a2927']
Response status: completed
Response output types: ['mcp_call', 'message']

Response output:
Story:
By day in the glass towers of Frankfurt, Germany, Cloud Solutions Architect - Security Azure Zo designed fortresses of encryption and zero‑trust networks while the River Main shimmered beneath the skyline.  
By night, Cloud Solutions Architect - Security Azure Zo became Die DEfender, a cybernetic guardian who patrolled the unseen circuitry of Europe’s digital infrastructure.  